[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

# Protocol Buffers Primer — Hands-On Application

| # | Section | Description |
|---|---------|-------------|
| 1 | [Setup](#1-setup) | Install dependencies, imports |
| 2 | [Exercise 1: Varint Encoder/Decoder](#2-exercise-1-varint-encoderdecoder) | Implement and test varint encoding |
| 3 | [Exercise 2: Wire Format Dissector](#3-exercise-2-wire-format-dissector) | Parse raw protobuf bytes |
| 4 | [Exercise 3: Explore onnx.proto](#4-exercise-3-explore-onnxproto) | Navigate the ONNX schema |
| 5 | [Exercise 4: Build and Serialize Models](#5-exercise-4-build-and-serialize-models) | End-to-end protobuf lifecycle |
| 6 | [Exercise 5: Size Scaling Experiment](#6-exercise-5-size-scaling-experiment) | Measure size vs complexity |
| 7 | [Exercise 6: Field Presence and Defaults](#7-exercise-6-field-presence-and-defaults) | Proto3 default value behavior |
| 8 | [Exercise 7: Visualize Byte Distribution](#8-exercise-7-visualize-byte-distribution) | Analyze serialized byte patterns |
| 9 | [Challenge: Build a Protobuf Hex Viewer](#9-challenge-build-a-protobuf-hex-viewer) | Annotated hex dump tool |
| 10 | [Summary](#10-summary) | Review of skills practiced |

In [ ]:
# !pip install onnx numpy matplotlib --quiet

import onnx
from onnx import helper, TensorProto, checker, numpy_helper
import numpy as np
import os
import json
import matplotlib.pyplot as plt

## 2. Exercise 1: Varint Encoder/Decoder

Implement the Protobuf varint encoding from scratch. Recall the encoding formula:

$$\text{value} = \sum_{i=0}^{n-1} (b_i \;\&\; \texttt{0x7F}) \ll (7i)$$

Each byte contributes 7 payload bits; the MSB is a continuation flag ($1$ = more bytes follow, $0$ = last byte).

**Tasks:**
1. Write `encode_varint(value)` that returns bytes
2. Write `decode_varint(data)` that returns (value, bytes_consumed)
3. Verify round-trip for edge cases: 0, 1, 127, 128, 300, $2^{63}-1$

In [ ]:
def encode_varint(value: int) -> bytes:
    """Encode a non-negative integer as a Protobuf varint."""
    if value == 0:
        return b'\x00'
    result = bytearray()
    while value > 0x7F:
        result.append((value & 0x7F) | 0x80)
        value >>= 7
    result.append(value & 0x7F)
    return bytes(result)


def decode_varint(data: bytes) -> tuple:
    """Decode a varint, return (value, bytes_consumed)."""
    value = 0
    for i, byte in enumerate(data):
        value |= (byte & 0x7F) << (7 * i)
        if not (byte & 0x80):
            return value, i + 1
    raise ValueError("Truncated varint")


# Test suite
test_cases = [0, 1, 127, 128, 255, 300, 16383, 16384, 2**20, 2**32 - 1, 2**63 - 1]
print(f"{'Value':>22} │ {'Encoded (hex)':>24} │ {'Bytes':>5} │ {'Round-trip':>10}")
print("─" * 72)
for v in test_cases:
    enc = encode_varint(v)
    dec, consumed = decode_varint(enc)
    ok = "✓" if dec == v else "✗"
    print(f"{v:>22,} │ {enc.hex():>24} │ {len(enc):>5} │ {ok:>10}")

## 3. Exercise 2: Wire Format Dissector

Build a basic Protobuf wire format parser that can dissect raw serialized bytes into `(field_number, wire_type, raw_value)` triples.

Recall: each field is stored as a **(tag, value)** pair where:
$$\text{tag} = (\text{field\_number} \ll 3) \;|\; \text{wire\_type}$$

Wire types determine how to read the value:
- $0$ (Varint): read a varint
- $1$ (64-bit): read 8 bytes
- $2$ (Length-delimited): read varint length, then that many bytes
- $5$ (32-bit): read 4 bytes

In [ ]:
def dissect_protobuf(data: bytes) -> list:
    """Parse raw protobuf bytes into (field_number, wire_type, value) triples."""
    fields = []
    pos = 0
    wire_names = {0: 'Varint', 1: '64-bit', 2: 'LenDel', 5: '32-bit'}
    
    while pos < len(data):
        tag, consumed = decode_varint(data[pos:])
        pos += consumed
        field_num = tag >> 3
        wire_type = tag & 0x07
        
        if wire_type == 0:  # Varint
            value, consumed = decode_varint(data[pos:])
            pos += consumed
            fields.append((field_num, wire_names.get(wire_type, '?'), value))
        elif wire_type == 1:  # 64-bit
            value = data[pos:pos+8]
            pos += 8
            fields.append((field_num, wire_names.get(wire_type, '?'), value.hex()))
        elif wire_type == 2:  # Length-delimited
            length, consumed = decode_varint(data[pos:])
            pos += consumed
            value = data[pos:pos+length]
            pos += length
            try:
                text = value.decode('utf-8')
                fields.append((field_num, wire_names.get(wire_type, '?'), f'str:"{text}"' if text.isprintable() else f'bytes[{length}]'))
            except (UnicodeDecodeError, ValueError):
                fields.append((field_num, wire_names.get(wire_type, '?'), f'bytes[{length}]'))
        elif wire_type == 5:  # 32-bit
            value = data[pos:pos+4]
            pos += 4
            fields.append((field_num, wire_names.get(wire_type, '?'), value.hex()))
        else:
            break
    return fields


# Dissect a simple ONNX model
simple_node = helper.make_node("Relu", ["X"], ["Y"])
simple_graph = helper.make_graph(
    [simple_node], "test",
    [helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, 4])],
    [helper.make_tensor_value_info("Y", TensorProto.FLOAT, [1, 4])],
)
simple_model = helper.make_model(simple_graph, opset_imports=[helper.make_opsetid("", 17)])
raw = simple_model.SerializeToString()

print(f"Serialized model: {len(raw)} bytes\n")
print(f"{'Field#':>6} │ {'Wire Type':>10} │ Value")
print("─" * 60)
for fn, wt, val in dissect_protobuf(raw):
    print(f"{fn:>6} │ {wt:>10} │ {val}")

## 4. Exercise 3: Explore onnx.proto

The ONNX proto file ships with the Python package. Let's parse it to extract the message hierarchy and field definitions. This exercise builds intuition for how the schema maps to the Python API.

**Tasks:**
1. Locate `onnx.proto` in your environment
2. Extract all message names and their field counts
3. Identify which messages use `repeated` fields (these become Python lists)

In [ ]:
import re

proto_path = os.path.join(os.path.dirname(onnx.__file__), 'onnx.proto')

if os.path.exists(proto_path):
    with open(proto_path) as f:
        proto_content = f.read()
    
    # Extract message definitions with field counts
    msg_pattern = re.compile(r'message\s+(\w+)\s*\{([^}]*(?:\{[^}]*\}[^}]*)*)\}', re.DOTALL)
    field_pattern = re.compile(r'^\s+(?:repeated\s+|optional\s+)?\w+\s+\w+\s*=\s*(\d+)', re.MULTILINE)
    repeated_pattern = re.compile(r'^\s+repeated\s+', re.MULTILINE)
    
    print(f"{'Message Name':<35} │ {'Fields':>6} │ {'Repeated':>8}")
    print("─" * 58)
    for match in re.finditer(r'message\s+(\w+)', proto_content):
        name = match.group(1)
        start = match.end()
        # Count fields (lines with = N;)
        depth = 0
        end = start
        for i in range(start, len(proto_content)):
            if proto_content[i] == '{':
                depth += 1
            elif proto_content[i] == '}':
                depth -= 1
                if depth == 0:
                    end = i
                    break
        body = proto_content[start:end]
        fields = len(re.findall(r'=\s*\d+\s*;', body))
        repeats = len(re.findall(r'repeated\s+', body))
        print(f"{name:<35} │ {fields:>6} │ {repeats:>8}")
else:
    print("onnx.proto not found — install onnx with pip install onnx")

In [ ]:
# Show the first 40 lines of onnx.proto
if os.path.exists(proto_path):
    with open(proto_path) as f:
        lines = f.readlines()[:40]
    print("First 40 lines of onnx.proto:")
    print("─" * 60)
    for i, line in enumerate(lines, 1):
        print(f"{i:3d} │ {line}", end='')

## 5. Exercise 4: Build and Serialize Models

Practice the full Protobuf lifecycle: construct a model object, serialize to bytes, inspect the binary, and deserialize back.

**Lifecycle:**
```
┌──────────────┐   SerializeToString()   ┌─────────────┐
│  ModelProto   │───────────────────────▶│  raw bytes   │
│  (in memory)  │                        │              │
└──────────────┘                         └─────────────┘
       ▲                                       │
       │         ParseFromString()             │
       └───────────────────────────────────────┘
```

**Task:** Build a model with 3 chained operations: $Y = \text{Sigmoid}(\text{MatMul}(X, W) + b)$

In [ ]:
# Build: Y = Sigmoid(MatMul(X, W) + b)
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", 4])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", 3])

W_init = numpy_helper.from_array(np.random.randn(4, 3).astype(np.float32), name="W")
b_init = numpy_helper.from_array(np.zeros(3, dtype=np.float32), name="b")

nodes = [
    helper.make_node("MatMul", ["X", "W"], ["mm_out"]),
    helper.make_node("Add", ["mm_out", "b"], ["add_out"]),
    helper.make_node("Sigmoid", ["add_out"], ["Y"]),
]

graph = helper.make_graph(nodes, "sigmoid_linear", [X], [Y], initializer=[W_init, b_init])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
model.producer_name = "protobuf_exercise"
model.doc_string = "Y = Sigmoid(X @ W + b)"
checker.check_model(model)

raw = model.SerializeToString()
print(f"Model built successfully!")
print(f"  Nodes: {[n.op_type for n in model.graph.node]}")
print(f"  Serialized size: {len(raw)} bytes")
print(f"  Weight tensor W: shape={list(W_init.dims)}, {np.prod(list(W_init.dims))*4} bytes of float data")

# Round-trip
restored = onnx.ModelProto()
restored.ParseFromString(raw)
assert restored.graph.name == model.graph.name
assert len(restored.graph.node) == 3
assert restored.SerializeToString() == raw
print("\nRound-trip: byte-identical ✓")

## 6. Exercise 5: Size Scaling Experiment

How does serialized model size grow with the number of parameters? Let's measure empirically and compare to the theoretical lower bound.

For a model with $N$ float32 parameters, the theoretical minimum size is $4N$ bytes (raw weight data). Protobuf adds overhead for tags, field lengths, and graph metadata.

**Overhead ratio:**
$$\text{overhead} = \frac{\text{total\_bytes} - 4N}{4N} \times 100\%$$

In [ ]:
def build_weighted_model(n_params: int) -> onnx.ModelProto:
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, n_params])
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [1, n_params])
    W = numpy_helper.from_array(np.random.randn(n_params).astype(np.float32), name="W")
    node = helper.make_node("Mul", ["X", "W"], ["Y"])
    graph = helper.make_graph([node], "scaled", [X], [Y], initializer=[W])
    return helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

param_sizes = [10, 50, 100, 500, 1000, 5000, 10000, 50000, 100000]
results = []

print(f"{'N params':>10} │ {'Proto (B)':>12} │ {'Raw 4N (B)':>12} │ {'JSON (B)':>12} │ {'Overhead':>10} │ {'JSON/PB':>8}")
print("─" * 78)
for n in param_sizes:
    m = build_weighted_model(n)
    pb_size = len(m.SerializeToString())
    raw_size = 4 * n
    overhead = (pb_size - raw_size) / raw_size * 100
    
    j = json.dumps({"w": numpy_helper.to_array(m.graph.initializer[0]).tolist()})
    json_size = len(j.encode())
    
    results.append((n, pb_size, raw_size, json_size, overhead))
    print(f"{n:>10,} │ {pb_size:>12,} │ {raw_size:>12,} │ {json_size:>12,} │ {overhead:>9.1f}% │ {json_size/pb_size:>7.1f}x")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ns = [r[0] for r in results]
pb = [r[1] for r in results]
raw = [r[2] for r in results]
js = [r[3] for r in results]
overheads = [r[4] for r in results]

ax = axes[0]
ax.loglog(ns, pb, 'o-', label='Protobuf', color='#2196F3', linewidth=2)
ax.loglog(ns, raw, 's--', label='Raw 4N bytes', color='#4CAF50', linewidth=2)
ax.loglog(ns, js, '^-', label='JSON', color='#FF9800', linewidth=2)
ax.set_xlabel('Number of Parameters', fontsize=12)
ax.set_ylabel('Serialized Size (bytes)', fontsize=12)
ax.set_title('Size Scaling: Protobuf vs JSON vs Raw', fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.semilogx(ns, overheads, 'o-', color='#F44336', linewidth=2, markersize=8)
ax.set_xlabel('Number of Parameters', fontsize=12)
ax.set_ylabel('Protobuf Overhead over Raw 4N (%)', fontsize=12)
ax.set_title('Protobuf Overhead Decreases with Scale', fontweight='bold')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"At 100K params, Protobuf overhead is only {overheads[-1]:.1f}% above the raw weight data.")

## 7. Exercise 6: Field Presence and Defaults

In proto3, fields that equal the default value ($0$ for numbers, `""` for strings, empty for repeated) are **not serialized** — they contribute zero bytes. This is different from proto2 where `optional` fields have explicit presence.

**Implication for ONNX:** When you read `model.model_version` and get $0$, you can't distinguish between "the field was explicitly set to 0" and "the field was never set." For string fields like `doc_string`, an empty string means "absent."

**Task:** Create models with and without optional fields set and observe the size differences.

In [ ]:
def make_minimal_model():
    node = helper.make_node("Relu", ["X"], ["Y"])
    graph = helper.make_graph(
        [node], "g",
        [helper.make_tensor_value_info("X", TensorProto.FLOAT, [1])],
        [helper.make_tensor_value_info("Y", TensorProto.FLOAT, [1])],
    )
    return helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

# Minimal model
m_min = make_minimal_model()
s_min = len(m_min.SerializeToString())

# With producer name
m_prod = make_minimal_model()
m_prod.producer_name = "my_converter"
s_prod = len(m_prod.SerializeToString())

# With doc string
m_doc = make_minimal_model()
m_doc.doc_string = "A simple Relu model for testing purposes"
s_doc = len(m_doc.SerializeToString())

# With all metadata
m_full = make_minimal_model()
m_full.producer_name = "my_converter"
m_full.producer_version = "2.0.1"
m_full.domain = "com.example"
m_full.model_version = 42
m_full.doc_string = "A simple Relu model for testing purposes"
entry = m_full.metadata_props.add()
entry.key = "author"
entry.value = "tutorial"
s_full = len(m_full.SerializeToString())

print(f"{'Configuration':<40} │ {'Size (B)':>10} │ {'Delta':>8}")
print("─" * 65)
print(f"{'Minimal (no optional fields)':<40} │ {s_min:>10} │ {'—':>8}")
print(f"{'+ producer_name':<40} │ {s_prod:>10} │ {s_prod-s_min:>+8}")
print(f"{'+ doc_string':<40} │ {s_doc:>10} │ {s_doc-s_min:>+8}")
print(f"{'All metadata set':<40} │ {s_full:>10} │ {s_full-s_min:>+8}")
print(f"\nDefault-valued fields cost 0 bytes — proto3 simply omits them.")

## 8. Exercise 7: Visualize Byte Distribution

What does the inside of a serialized ONNX model look like? Let's analyze the byte value distribution to understand the structure.

In a model dominated by float32 weights, we expect a roughly uniform byte distribution (random weight values). In a metadata-heavy model, we'll see more ASCII-range bytes (0x20–0x7E) from strings.

In [ ]:
# Build a model with substantial weights
large_model = build_weighted_model(10000)
large_bytes = large_model.SerializeToString()

# Byte frequency histogram
byte_counts = np.zeros(256, dtype=int)
for b in large_bytes:
    byte_counts[b] += 1

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Full histogram
ax = axes[0]
ax.bar(range(256), byte_counts, color='#2196F3', alpha=0.7, width=1.0)
ax.axvspan(0x20, 0x7E, alpha=0.1, color='green', label='ASCII printable range')
ax.set_xlabel('Byte Value (0x00 — 0xFF)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title(f'Byte Distribution in Serialized Model ({len(large_bytes):,} bytes)', fontweight='bold')
ax.legend()
ax.set_xlim(-1, 256)

# Categorized pie chart
ax = axes[1]
categories = {
    'Null (0x00)': byte_counts[0],
    'Control (0x01-0x1F)': byte_counts[1:0x20].sum(),
    'ASCII printable': byte_counts[0x20:0x7F].sum(),
    'High bytes (0x80-0xFF)': byte_counts[0x80:].sum(),
}
colors = ['#F44336', '#FF9800', '#4CAF50', '#2196F3']
wedges, texts, autotexts = ax.pie(
    categories.values(), labels=categories.keys(), autopct='%1.1f%%',
    colors=colors, startangle=90
)
ax.set_title('Byte Category Distribution', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"High bytes (0x80+) are dominant because float32 weight data")
print(f"produces roughly uniform byte values across the full 0-255 range.")

## 9. Challenge: Build a Protobuf Hex Viewer

Create an annotated hex dump that labels each section of a serialized ONNX model — identifying where tags, field values, and embedded messages begin and end.

This is a deeper exercise that ties together all the wire format concepts:
- Tag decoding: $(\text{field\_number}, \text{wire\_type})$
- Varint decoding for lengths and values
- Nested message boundaries

In [ ]:
def annotated_hex_dump(data: bytes, max_bytes: int = 120) -> None:
    """Print an annotated hex dump of protobuf data."""
    wire_names = {0: 'VARINT', 1: 'I64', 2: 'LEN', 5: 'I32'}
    pos = 0
    field_num_counter = 0
    
    print(f"{'Offset':>6} │ {'Hex':>20} │ {'Field':>6} │ {'Wire':>6} │ Description")
    print("─" * 80)
    
    while pos < min(len(data), max_bytes):
        start = pos
        tag, consumed = decode_varint(data[pos:])
        pos += consumed
        field_num = tag >> 3
        wire_type = tag & 0x07
        
        tag_hex = data[start:pos].hex()
        wt_name = wire_names.get(wire_type, '???')
        
        if wire_type == 0:
            val_start = pos
            value, consumed = decode_varint(data[pos:])
            pos += consumed
            val_hex = data[val_start:pos].hex()
            print(f"{start:>6} │ {tag_hex + ' ' + val_hex:>20} │ {field_num:>6} │ {wt_name:>6} │ value = {value}")
        elif wire_type == 2:
            len_start = pos
            length, consumed = decode_varint(data[pos:])
            pos += consumed
            payload = data[pos:pos+length]
            pos += length
            try:
                text = payload.decode('utf-8')
                if text.isprintable() and len(text) < 40:
                    desc = f'string: "{text}"'
                else:
                    desc = f'bytes[{length}] (sub-message or binary)'
            except (UnicodeDecodeError, ValueError):
                desc = f'bytes[{length}] (sub-message or binary)'
            len_hex = data[len_start:len_start+consumed].hex()
            print(f"{start:>6} │ {tag_hex + ' ' + len_hex + '...':>20} │ {field_num:>6} │ {wt_name:>6} │ len={length}, {desc}")
        elif wire_type == 5:
            val_hex = data[pos:pos+4].hex()
            pos += 4
            print(f"{start:>6} │ {tag_hex + ' ' + val_hex:>20} │ {field_num:>6} │ {wt_name:>6} │ 32-bit value")
        elif wire_type == 1:
            val_hex = data[pos:pos+8].hex()
            pos += 8
            print(f"{start:>6} │ {tag_hex + ' ' + val_hex:>20} │ {field_num:>6} │ {wt_name:>6} │ 64-bit value")
        else:
            print(f"{start:>6} │ {'???':>20} │ {field_num:>6} │ {'???':>6} │ Unknown wire type {wire_type}")
            break
    
    if pos < len(data):
        print(f"  ... ({len(data) - pos} more bytes not shown)")


# Use the simple model from earlier
print(f"Annotated hex dump of a Relu model ({len(simple_model.SerializeToString())} bytes):\n")
annotated_hex_dump(simple_model.SerializeToString(), max_bytes=200)

In [ ]:
# Save/load file round-trip
import tempfile

with tempfile.NamedTemporaryFile(suffix='.onnx', delete=False) as f:
    tmp_path = f.name

onnx.save_model(model, tmp_path)
file_size = os.path.getsize(tmp_path)
loaded = onnx.load_model(tmp_path)
checker.check_model(loaded)

print(f"Saved to:     {tmp_path}")
print(f"File size:    {file_size} bytes")
print(f"Graph name:   {loaded.graph.name}")
print(f"Nodes:        {[n.op_type for n in loaded.graph.node]}")
print(f"Match:        {loaded.SerializeToString() == model.SerializeToString()}")

os.unlink(tmp_path)
print("Cleanup done.")

## 10. Summary

In this hands-on notebook you have:

1. **Implemented varint encoding/decoding** from scratch using the formula $\text{value} = \sum (b_i \;\&\; \texttt{0x7F}) \ll 7i$ and verified round-trip correctness for edge cases

2. **Built a wire format dissector** that parses raw Protobuf bytes into `(field_number, wire_type, value)` triples, understanding the tag encoding $\text{tag} = (\text{fn} \ll 3) | \text{wt}$

3. **Explored `onnx.proto`** to understand the message hierarchy that defines the ONNX model format

4. **Practiced the full serialization lifecycle**: build → serialize → inspect bytes → deserialize → verify

5. **Measured size scaling** empirically, confirming Protobuf's overhead converges to near-zero as model parameters dominate

6. **Analyzed byte distributions** to understand what serialized ONNX models look like at the binary level

7. **Built a hex viewer** that annotates Protobuf wire format structures in serialized model data